In [4]:
import pandas as pd
from datasets import load_dataset
from functions.pred import *
from functions.xai import *
from functions.eval import *
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [ ]:
base = "https://huggingface.co/datasets/cardiffnlp/tweet_sentiment_multilingual/resolve/refs%2Fconvert%2Fparquet/arabic"
train_df = pd.read_parquet(f"{base}/train/0000.parquet")
test_df  = pd.read_parquet(f"{base}/test/0000.parquet")
val_df   = pd.read_parquet(f"{base}/validation/0000.parquet")

In [15]:
df = pd.concat([train_df, test_df, val_df], ignore_index=True)
df

,text,label
0,RT @user: @user @user وصلنا لاقتصاد اسوء من ...,0
1,كاني ويست، دريك، نيكي، بيونسيه، قاقا http,1
2,@user على فكره شركة محترمه حداعطوني كيبل كهديه...,2
3,RT @user: المتعه افضل من الزواج لهتك اعراض عام...,0
4,القوات البرية السعودية والقوات الفرنسية الخاصة...,1
...,...,...
3028,قال رسول الله ﷺ(إذا سمعتم الطاعون بأرض، فلا تد...,1
3029,RT @user: ماركا | لم ينهزم ريال مدريد في أخر 2...,2
3030,RT @user: #مليارات_العمره_على_هوى_مصر سقوط بشا...,0
3031,حد معاه ويندوز 10 ؟,1


In [16]:
df["label"] = df["label"].map({0: "negative", 1: "neutral", 2: "positive"})

In [18]:
# removing @user, RT, http from the text
df["text"] = df["text"].str.replace("@user", "")
df["text"] = df["text"].str.replace("RT", "")
df["text"] = df["text"].str.replace("http", "")

In [19]:
# remove multiple spaces
df["text"] = df["text"].str.replace("\s+", " ", regex=True)
df["text"] = df["text"].str.strip()

<>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
C:\Users\user\AppData\Local\Temp\ipykernel_4700\3601961224.py:2: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  df["text"] = df["text"].str.replace("\s+", " ", regex=True)


In [21]:
df["text"] = df["text"].apply(remove_chaklas)

In [23]:
df.label.value_counts()

label
negative    1011
neutral     1011
positive    1011
Name: count, dtype: int64

In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "CAMeL-Lab/bert-base-arabic-camelbert-msa-sentiment"
pipe = pipeline("text-classification", model=model_name, top_k=None, device=device)

Device set to use cuda


In [25]:
df["Camelbert-MSA"] = df["text"].apply(lambda x: predict_class(pipe, x))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "PRAli22/AraBert-Arabic-Sentiment-Analysis"
pipe = pipeline("text-classification", model=model_name, top_k=None, device=device)

config.json:   0%|          | 0.00/941 [00:00<?, ?B/s]

c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--PRAli22--AraBert-Arabic-Sentiment-Analysis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not inst

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 6554e590-7c57-46d7-bab8-827c7c211186)')' thrown while requesting HEAD https://huggingface.co/PRAli22/AraBert-Arabic-Sentiment-Analysis/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Device set to use cuda


In [27]:
df["AraBert"] = df["text"].apply(lambda x: predict_class(pipe, x))

In [28]:
accuracy = accuracy_score(df["label"], df["Camelbert-MSA"])
precision = precision_score(df["label"], df["Camelbert-MSA"], average="weighted")
recall = recall_score(df["label"], df["Camelbert-MSA"], average="weighted")
f1 = f1_score(df["label"], df["Camelbert-MSA"], average="weighted")
print("Camelbert-MSA")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")

Camelbert-MSA
Accuracy: 0.7781074843389384
Precision: 0.7853841647889654
Recall: 0.7781074843389384
F1-score: 0.7791765256324032


In [29]:
accuracy = accuracy_score(df["label"], df["AraBert"].str.lower())
precision = precision_score(df["label"], df["AraBert"].str.lower(), average="weighted")
recall = recall_score(df["label"], df["AraBert"].str.lower(), average="weighted")
f1 = f1_score(df["label"], df["AraBert"].str.lower(), average="weighted")
print("AraBert")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")

AraBert
Accuracy: 0.6007253544345532
Precision: 0.6500046799705592
Recall: 0.6007253544345532
F1-score: 0.6051744879451739


c:\Users\user\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [30]:
df

,text,label,Camelbert-MSA,AraBert
0,: وصلنا لاقتصاد اسوء من سوريا والعراق ومن غير ...,negative,negative,Negative
1,كاني ويست، دريك، نيكي، بيونسيه، قاقا,neutral,neutral,Neutral
2,على فكره شركة محترمه حداعطوني كيبل كهديه ويوم ...,positive,positive,Negative
3,: المتعه افضل من الزواج لهتك اعراض عامة #الشيع...,negative,negative,Negative
4,القوات البرية السعودية والقوات الفرنسية الخاصة...,neutral,neutral,Neutral
...,...,...,...,...
3028,قال رسول الله ﷺ(إذا سمعتم الطاعون بأرض، فلا تد...,neutral,neutral,Neutral
3029,: ماركا | لم ينهزم ريال مدريد في أخر 23 مباراة...,positive,positive,Neutral
3030,: #مليارات_العمره_على_هوى_مصر سقوط بشار الكلب ...,negative,negative,Negative
3031,حد معاه ويندوز 10 ؟,neutral,neutral,Negative


In [ ]:
df.to_csv("data/model_pred/model_pred.csv", index=False)